# GraphMind — Fine-Tuning Qwen2.5-7B-Instruct with Unsloth (LoRA/PEFT)

> **Capstone Project 1 — C1SE.18 — Duy Tan University**  
> **Research Track:** Parameter-Efficient Fine-Tuning (PEFT) on Kaggle T4 GPU (16GB VRAM)  
> **Mục tiêu:** Huấn luyện mô hình Qwen2.5-7B thành thạo cú pháp **Text-to-Cypher** trên Neo4j và tổng hợp câu trả lời **RAG có trích dẫn nguồn (Grounded Citations)** từ CUAD & AdventureWorks 100% dữ liệu thực.

## 1. Cài đặt Unsloth & Thư viện phụ thuộc (Tối ưu 4x tốc độ & giảm 70% VRAM)

In [ ]:
%%capture
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps trl peft accelerate bitsandbytes datasets

## 2. Tải Mô hình Nền Qwen2.5-7B-Instruct (4-bit Quantization)

In [ ]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 2048
dtype = None # Auto detect: Float16 cho T4
load_in_4bit = True # Tiết kiệm VRAM dưới 6GB

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Qwen2.5-7B-Instruct-bnb-4bit",
    max_seq_length=max_seq_length,
    dtype=dtype,
    load_in_4bit=load_in_4bit,
)
print("[OK] Base model Qwen2.5-7B loaded successfully!")

## 3. Cấu hình LoRA Adapters (PEFT)

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
)
model.print_trainable_parameters()

## 4. Tải Tập Dữ liệu Huấn luyện Thật (train_dataset.json)

In [ ]:
from datasets import Dataset
import json

# Đọc 509 mẫu dữ liệu sinh từ đồ thị thực tế GraphMind
dataset_path = "train_dataset.json"
with open(dataset_path, "r", encoding="utf-8") as f:
    raw_data = json.load(f)

# Định dạng Qwen2.5 Chat Template (ChatML)
def formatting_prompts_func(examples):
    texts = []
    for inst, inp, out in zip(examples["instruction"], examples["input"], examples["output"]):
        user_msg = f"{inst}\n{inp}".strip()
        text = f"<|im_start|>system\nBạn là trợ lý AI GraphMind, hỗ trợ truy vấn đồ thị tri thức doanh nghiệp.<|im_end|>\n<|im_start|>user\n{user_msg}<|im_end|>\n<|im_start|>assistant\n{out}<|im_end|>"
        texts.append(text)
    return {"text": texts}

hf_dataset = Dataset.from_list(raw_data)
hf_dataset = hf_dataset.map(formatting_prompts_func, batched=True)
print(f"[OK] Dataset ready with {len(hf_dataset)} instruction samples.")
print("Sample prompt:\n", hf_dataset[0]["text"][:300])

## 5. Tiến hành Huấn luyện (SFTTrainer)

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=hf_dataset,
    dataset_text_field="text",
    max_seq_length=max_seq_length,
    dataset_num_proc=2,
    packing=False,
    args=TrainingArguments(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_steps=10,
        max_steps=60, # Tinh chỉnh nhanh (~15 phút trên Kaggle T4)
        learning_rate=2e-4,
        fp16=not is_bfloat16_supported(),
        bf16=is_bfloat16_supported(),
        logging_steps=5,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="linear",
        seed=42,
        output_dir="outputs",
    ),
)

trainer_stats = trainer.train()
print("[OK] Fine-tuning completed successfully!")

## 6. Kiểm thử Suy luận (Inference Test)

In [ ]:
FastLanguageModel.for_inference(model)

prompt = "<|im_start|>system\nBạn là trợ lý AI GraphMind, hỗ trợ truy vấn đồ thị tri thức doanh nghiệp.<|im_end|>\n<|im_start|>user\nChuyển câu hỏi sau thành câu lệnh Cypher: 'Tìm điều khoản Termination trong hợp đồng của công ty WPP PLC'<|im_end|>\n<|im_start|>assistant\n"
inputs = tokenizer([prompt], return_tensors="pt").to("cuda")
outputs = model.generate(**inputs, max_new_tokens=128, temperature=0.2)
print(tokenizer.decode(outputs[0], skip_special_tokens=True))

## 7. Xuất Định dạng GGUF cho Ollama Local Runtime

In [ ]:
# Lưu trực tiếp thành file GGUF 4-bit tương thích hoàn hảo với Ollama
model.save_pretrained_gguf("qwen2.5-7b-graphmind", tokenizer, quantization_method="q4_k_m")

# Tạo file Modelfile mẫu cho Ollama
modelfile_content = """FROM ./qwen2.5-7b-graphmind-unsloth.Q4_K_M.gguf
PARAMETER temperature 0.3
PARAMETER top_p 0.9
PARAMETER stop "<|im_end|>"
SYSTEM Bạn là trợ lý thông minh GraphMind (AEGIS Enterprise Knowledge Platform).
"""
with open("Modelfile", "w") as f:
    f.write(modelfile_content)
print("[OK] GGUF and Modelfile exported. To import to local Ollama run:")
print("  ollama create qwen2.5-7b-graphmind -f Modelfile")